# 01 · Exploratory Data Analysis — Heart Disease UCI

**Assignment:** AIMLCZG523 MLOps Assignment 01 · Task 1 (5 marks)

This notebook performs a professional EDA of the merged 4-hospital Heart Disease dataset.

1. Load raw CSV
2. Schema, missing value analysis
3. Class balance
4. Distributions (histograms + KDEs)
5. Correlation heatmap
6. Feature-target relationships

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if (ROOT / 'src').is_dir():
    PROJECT_ROOT = ROOT
else:
    PROJECT_ROOT = ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data_preprocessing import load_raw_dataset, prepare_dataset

sns.set_theme(context='notebook', style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

FIG_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

## 1. Load & inspect raw dataset

In [ ]:
df = load_raw_dataset()
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 2. Missing value analysis

In [ ]:
missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing_df = missing.reset_index()
missing_df.columns = ['column', 'missing_pct']

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=missing_df, x='missing_pct', y='column', ax=ax, color='#4C72B0')
ax.set_title('Missing value percentage by column')
ax.set_xlabel('Missing %')
ax.set_ylabel('')
fig.tight_layout()
fig.savefig(FIG_DIR / 'eda_missing_values.png')
plt.show()
missing_df

## 3. Engineer binary target & inspect class balance

In [ ]:
X, y = prepare_dataset(df)
print('Feature matrix:', X.shape)
print('Target class counts:')
print(y.value_counts().rename({0: 'No Disease', 1: 'Disease'}))

fig, ax = plt.subplots(figsize=(5.5, 4))
sns.countplot(x=y.map({0: 'No Disease', 1: 'Disease'}), ax=ax, palette='deep')
ax.set_title('Binary target class balance')
ax.set_xlabel('')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
fig.tight_layout()
fig.savefig(FIG_DIR / 'eda_class_balance.png')
plt.show()

## 4. Numeric feature distributions

In [ ]:
numeric_cols = config.NUMERIC_FEATURES
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax, color='#4C72B0')
    ax.set_title(f'Distribution of {col}')
fig.tight_layout()
fig.savefig(FIG_DIR / 'eda_numeric_histograms.png')
plt.show()

## 5. Categorical feature vs target

In [ ]:
cats = [c for c in config.CATEGORICAL_FEATURES if c in df.columns]
plot_df = df.copy()
plot_df['target'] = (plot_df['num'] >= 1).astype(int).map({0: 'No Disease', 1: 'Disease'})

n = len(cats)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
for ax, col in zip(axes.flatten(), cats):
    ct = pd.crosstab(plot_df[col], plot_df['target'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=ax, color=['#55A868', '#C44E52'])
    ax.set_title(f'{col} vs target (%)')
    ax.set_ylabel('% of rows')
    ax.legend(title='')
for ax in axes.flatten()[len(cats):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'eda_categorical_vs_target.png')
plt.show()

## 6. Correlation heatmap (numeric features + target)

In [ ]:
corr_df = df[numeric_cols].copy()
corr_df['target'] = (df['num'] >= 1).astype(int)
corr = corr_df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation heatmap (numeric features + target)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'eda_correlation_heatmap.png')
plt.show()

## 7. Key EDA takeaways

- The dataset contains **920 patients** from four hospitals. The binary target derived from `num >= 1` is only mildly imbalanced (~55% disease).
- Several columns have substantial missingness (`ca`, `thal`, `slope`, `chol`), which drives our choice of median / most-frequent imputers inside the sklearn `Pipeline`.
- `oldpeak`, `thalch`, `age` and `ca` show the strongest linear relationship with the target.
- Categorical features `cp` (chest-pain type), `exang` (exercise-induced angina) and `thal` are highly discriminative.
- The `dataset` column encodes hospital origin; we retain it since disease prevalence differs across hospitals.